### Load GS Data to HDFS


In [ ]:
# Drop existing table if any, then create an external table `wdi_csv_text`
# Data is CSV format with comma delimiters, stored at an HDFS location.

DROP TABLE IF EXISTS wdi_csv_text;
CREATE EXTERNAL TABLE wdi_csv_text
(year INTEGER, countryName STRING, countryCode STRING, indicatorName STRING, indicatorCode STRING, indicatorValue FLOAT)
ROW FORMAT DELIMITED FIELDS TERMINATED BY ',' LINES TERMINATED BY '\n'
LOCATION 'hdfs:///user/adityakhajanchi07/hive/wdi/wdi_csv_text';

In [ ]:
# Overwrite data in wdi_csv_text with all records from wdi_gs table

INSERT OVERWRITE TABLE wdi_csv_text
SELECT * FROM wdi_gs

In [ ]:
# Run twice to observe filesystem caching effects

SELECT Count(countryname) FROM wdi_csv_text;

**Observations after running identical query twice.**

First run took `28.502 seconds` to generate the output

Second run took `22.010 seconds` for the same query.


In [ ]:
# Clear filesystem cache and re-run count query to compare performance

SELECT count(countryName) FROM wdi_csv_text

**Third Run observations**

It took `36 seconds` for the system to run the same query.

Before the third run, I cleared the filesystem cache using the below command to see if there's any difference.


### Hive vs Bash: Query Processing Times


In [ ]:
%sh

cd ~
hdfs  dfs -get  hdfs:///user/adityakhajanchi07/hive/wdi/wdi_csv_text
cd wdi_csv_text

du -ch

echo 3 | sudo tee /proc/sys/vm/drop_caches

date +%s && cat * | wc && date +%s

TASK: Count the rows in the dataset.

HIVE: `36 seconds`
BASH: `17 seconds`

The delta is of a whopping `20 seconds`.

This is primarily due to their architectural design.

Bash operates directly on the filesystem with minimal overhead, making it quick for small and local tasks.

Hive, on the other hand goes through multiple layers to process the data. It involves parsing SQL, accessing metadata from metastore, applying schema checks and later triggering a MR | Tez job.

While Hive was slow for this task, it is optimal for a larger and complex data job.


### Parsing Issues


In [ ]:
# List distinct indicator codes, ordered alphabetically, limited to 20 results

SELECT distinct(indicatorcode)
FROM wdi_csv_text
ORDER BY indicatorcode
LIMIT 20;

# Observe the values under indicatorCode column

The above ouput indicates that there could be parsing issues. The next steps include diagnosing and debugging by creating a debug table.


In [ ]:
# Creating a new debug table with a single column

DROP TABLE IF EXISTS wdi_gs_debug;
CREATE EXTERNAL TABLE wdi_gs_debug (
  Column1 STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY '\b'
LINES TERMINATED BY '\n'
LOCATION 'gs://jarvis_data_eng_aditya/datasets/wdi_2016';

In [ ]:
SELECT * FROM wdi_gs_debug LIMIT 10;

In [ ]:
# Check rows containing '%( % of urban population)%' to verify parsing of indicatorCode values

SELECT * FROM wdi_gs_debug
WHERE column1 LIKE '%(% of urban population)%';

In [ ]:
# Create a Table with OpenCSV SerDe

DROP TABLE IF EXISTS wdi_opencsv_gs;

CREATE EXTERNAL TABLE wdi_opencsv_gs (
  year INT,
  countryName STRING,
  countryCode STRING,
  indicatorName STRING,
  indicatorCode STRING,
  indicatorValue FLOAT
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
  "separatorChar" = ",",
  "quoteChar" = "\""
)
STORED AS TEXTFILE
LOCATION 'gs://jarvis_data_eng_aditya/datasets/wdi_2016'
TBLPROPERTIES (
  "skip.header.line.count" = "1"
)

In [ ]:
# Verify if `wdi_opencsv_gs` table correctly parsed `indicatorCode` values using OpenCSVSerde

SELECT * FROM wdi_opencsv_gs LIMIT 10;

# Observe the values under `indicatorcode` column

In [ ]:
# Create wdi_opencsv_text destination table (output table with hdfs location)

DROP TABLE IF EXISTS wdi_opencsv_text;

CREATE EXTERNAL TABLE wdi_opencsv_text (
  year INT,
  countryName STRING,
  countryCode STRING,
  indicatorName STRING,
  indicatorCode STRING,
  indicatorValue FLOAT
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LINES TERMINATED BY '\n'
STORED AS TEXTFILE
LOCATION 'hdfs:///user/adityakhajanchi07/hive/wdi/wdi_opencsv_text'

In [ ]:
# Load data from `wdi_opencsv_gs` to `wdi_opencsv_text`

INSERT OVERWRITE TABLE wdi_opencsv_text
SELECT *
FROM wdi_opencsv_gs;

In [ ]:
# Verify parsing after loading data

SELECT * FROM wdi_opencsv_text LIMIT 10;

In [ ]:
# Ob parsing after loading data

SELECT distinct(indicatorcode) from wdi_opencsv_text limit 10;

Because `wdi_opencsv_text` uses standard `DELIMITED` formatting (not `OpenCSVSerde`), it writes data as raw text without quotes or escaping.

Hence, the resulting table again contains incorrectly parsed values in the `indicatorCode` column.


Compare execution time between `wdi_opencsv_text` and `wdi_csv_text`


In [ ]:
SELECT count(countryName) FROM wdi_opencsv_text

In [ ]:
# Observe the query processing time to check total number of records in the `wdi_csv_text` table

SELECT Count(countryname) FROM wdi_csv_text

#### Comparison of Execution Time: `wdi_opencsv_text` vs `wdi_csv_text`

It took `wdi_opencsv_text` **1m 19s 55ms** to complete the query.  
It took `wdi_csv_text` **23s 210ms** for the same operation.

There is a performance delta of **55s 340ms**, where `wdi_csv_text` performed significantly faster.  
This suggests that using a simpler SerDe (like `DELIMITED`) may be more efficient in certain contexts where complex parsing is unnecessary.


### OpenCSVSerde limitaion


In [ ]:
# View detailed metadata and storage information for the table `wdi_opencsv_gs`

DESCRIBE FORMATTED wdi_opencsv_gs;

In [ ]:
# Drop the view if it exists, then create a new view to simplify querying `wdi_opencsv_gs`

DROP VIEW IF EXISTS wdi_opencsv_text_view;

CREATE VIEW IF NOT EXISTS wdi_opencsv_text_view
AS
SELECT * FROM wdi_opencsv_gs;

### 2015 Canada GDP Growth HQL


In [ ]:
# HQL to find out the correct indicator name/code

SELECT DISTINCT indicatorName, indicatorCode
FROM wdi_opencsv_text
WHERE indicatorName LIKE '%GDP growth (annual %)%';

In [ ]:
# HQL to find 2015 Canada GDP growth

SELECT indicatorvalue AS GDP_growth_value, year as Year, countryName 
FROM wdi_opencsv_text 
WHERE countryName = 'Canada' 
AND year = 2015 
AND indicatorCode = 'NY.GDP.MKTP.KD.ZG';

#### Why did the HQL query took a significant time to process?

Since there were no partitions made on the table, Hive has to scan the entire dataset.


### Hive Partitions


In [ ]:
# Create wdi_opencsv_text_partitions table which partitioned by year

DROP TABLE IF EXISTS wdi_opencsv_text_partitions;

CREATE EXTERNAL TABLE wdi_opencsv_text_partitions (
  countryName STRING,
  countryCode STRING,
  indicatorName STRING,
  indicatorCode STRING,
  indicatorValue FLOAT
)
PARTITIONED BY (year INT)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LINES TERMINATED BY '\n'
STORED AS TEXTFILE
LOCATION 'hdfs:///user/adityakhajanchi07/hive/wdi/wdi_opencsv_text_partitions'

In [ ]:
# Load data from wdi_opencsv_text to wdi_opencsv_text_partitions using dynamic partitions

SET hive.exec.dynamic.partition = true;
SET hive.exec.dynamic.partition.mode = nonstrict;

INSERT OVERWRITE TABLE wdi_opencsv_text_partitions
PARTITION (year)
SELECT
  countryName,
  countryCode,
  indicatorName,
  indicatorCode,
  indicatorValue,
  year
FROM wdi_opencsv_text;

In [ ]:
%sh
# Observe the number of partitions created for wdi_opencsv_text_partitions table

hdfs dfs -ls hdfs:///user/adityakhajanchi07/hive/wdi/wdi_opencsv_text_partitions | grep "year=" | wc -l

In [ ]:
# Re-run 2015 GDP Growth HQL against the wdi_opencsv_text_partitions table and compare execution time the previous solution

SELECT indicatorvalue AS GDP_growth_value, year as Year, countryName 
FROM wdi_opencsv_text_partitions 
WHERE countryName = 'Canada' 
AND year = 2015 
AND indicatorCode = 'NY.GDP.MKTP.KD.ZG';

**From `1m 35s 866ms` to `2s`!!**  
Partitioning significantly reduced query execution time.


### Columnar File Optimization


In [ ]:
# Create a Parquet-backed table at the an HDFS location

DROP TABLE IF EXISTS wdi_csv_parquet;

CREATE EXTERNAL TABLE IF NOT EXISTS wdi_csv_parquet (
  year INT,
  countryName STRING,
  countryCode STRING,
  indicatorName STRING,
  indicatorCode STRING,
  indicatorValue FLOAT
)
STORED AS PARQUET
LOCATION 'hdfs:///user/adityakhajanchi07/hive/wdi/wdi_csv_parquet';

In [ ]:
# Load data from wdi_opencsv_gs to wdi_csv_parquet

INSERT OVERWRITE TABLE wdi_csv_parquet
SELECT *
FROM wdi_opencsv_gs;

In [ ]:
%sh
# Compare file sizes between wdi_csv_parquet and wdi_opencsv_text

hdfs dfs -du -h /user/adityakhajanchi07/hive/wdi

### Table Storage Size Comparison

| Table Name                    | Compressed Size | Uncompressed Size | HDFS Path                                                      |
| ----------------------------- | --------------- | ----------------- | -------------------------------------------------------------- |
| `wdi_csv_parquet`             | 131.9 MB        | 263.8 MB          | `/user/adityakhajanchi07/hive/wdi/wdi_csv_parquet`             |
| `wdi_csv_text`                | 1.7 GB          | 3.4 GB            | `/user/adityakhajanchi07/hive/wdi/wdi_csv_text`                |
| `wdi_gs`                      | 0               | 0                 | `/user/adityakhajanchi07/hive/wdi/wdi_gs`                      |
| `wdi_opencsv_text`            | 2.3 GB          | 4.5 GB            | `/user/adityakhajanchi07/hive/wdi/wdi_opencsv_text`            |
| `wdi_opencsv_text_partitions` | 1.9 GB          | 3.8 GB            | `/user/adityakhajanchi07/hive/wdi/wdi_opencsv_text_partitions` |


#### Compare query execution times for counting records in two tables to evaluate performance differences


In [ ]:
# Observe the query processing time to check total number of records in the `wdi_csv_parquet` table

SELECT Count(countryName) FROM wdi_csv_parquet;

In [ ]:
# Observe the query processing time to check total number of records in the `wdi_opencsv_text` table

SELECT count(countryName) FROM wdi_opencsv_text;

Counting rows in `wdi_csv_parquet` took 21 seconds, showcasing faster query performance due to Parquet's columnar storage and compression
Counting rows in `wdi_opencsv_text` took 1 minute 22 seconds, indicating slower performance caused by raw CSV format and lack of optimizations


In [ ]:
# Find the highest GDP growth value by year for each country
# Output: GDP_growth_value, year, countryName

SELECT indicatorvalue AS GDP_growth_value, year as Year, countryName 
FROM wdi_csv_parquet 
WHERE countryName = 'Canada' 
AND year = 2015 
AND indicatorCode = 'NY.GDP.MKTP.KD.ZG';

In [ ]:
# Find the highest GDP growth value by year for each country
# Output: GDP_growth_value, year, countryName

SELECT indicatorvalue AS GDP_growth_value, year as Year, countryName 
FROM wdi_opencsv_text
WHERE countryName = 'Canada' 
AND year = 2015 
AND indicatorCode = 'NY.GDP.MKTP.KD.ZG';

Query execution on `wdi_csv_parquet` is much faster (22s) than on `wdi_opencsv_text` (1m 22s), demonstrating the efficiency of Parquet's columnar storage and compression over CSV format.


### Highest GDP Growth


In [ ]:
# Highest GDP growth (NY.GDP.MKTP.KD.ZG) year for each country

SELECT countryName, countryCode, year, indicatorValue
FROM (
  SELECT 
    countryName,
    countryCode,
    year,
    indicatorValue,
    ROW_NUMBER() OVER (PARTITION BY countryCode ORDER BY indicatorValue DESC) as rank
  FROM wdi_csv_parquet
  WHERE indicatorCode = 'NY.GDP.MKTP.KD.ZG'
) ranked
WHERE rank = 1;

In [ ]:
%spark.sql
# Execute the same query using SparkSQL

SELECT countryName, countryCode, year, indicatorValue
FROM (
  SELECT 
    countryName,
    countryCode,
    year,
    indicatorValue,
    ROW_NUMBER() OVER (PARTITION BY countryCode ORDER BY indicatorValue DESC) as rank
  FROM wdi_csv_parquet
  WHERE indicatorCode = 'NY.GDP.MKTP.KD.ZG'
) ranked
WHERE rank = 1;

Query to find the highest GDP growth year per country using window functions.

`Hive` execution time: `37 seconds`, demonstrating efficient processing on Parquet with Hive.

`SparkSQL` execution time: `2 minutes 27 seconds`, slower than `Hive` for this specific analytical query on the same dataset.


### Sort GDP by country and year


In [ ]:
%hive
# GDP Growth for all coutries, sorted by countryName and year
SET hive.execution.engine=tez;

SELECT countryName, year, indicatorName, indicatorValue
FROM wdi_csv_parquet
WHERE indicatorCode = 'NY.GDP.MKTP.KD.ZG'
ORDER BY countryName, year;

Query processing time: `27s`
